# Comandos por Serial (cable USB) al ESP32 + PCA9685

Usa el mismo firmware `eris_servo_wifi.ino` (el Serial sigue activo junto con el WiFi), a **921600 baud**.
Tambien puede reproducir un CSV (celdas del final). Comandos: `X <canal> <ang>`, `X <a0> ... <a15>` (inmediato), `Y ...` (con rampa), `INFO`, `ON`, `OFF`.
Cierra el Monitor Serie del IDE antes de conectar (solo un programa puede usar el puerto).

In [ ]:
import serial, serial.tools.list_ports, time

for p in serial.tools.list_ports.comports():
    print(p.device, "-", p.description)

In [ ]:
PORT = "COM5"      # <- puerto de la lista de arriba
BAUD = 921600

ser = serial.Serial(PORT, BAUD, timeout=1)
time.sleep(2)             # el ESP32 se reinicia al abrir el puerto
ser.reset_input_buffer()  # descarta la linea "READY ..." del arranque

def cmd(text):
    """Envia una linea y devuelve la respuesta (una linea)."""
    ser.write((text.strip() + "\n").encode())
    return ser.readline().decode(errors="replace").strip()

print(cmd("INFO"))

In [ ]:
# Mover un canal (0-15) a un angulo (0-180), inmediato
print(cmd("X 0 90"))

In [ ]:
# Mover un canal con rampa suave (90 grados/s)
print(cmd("Y 0 45"))

In [ ]:
# Mover los 16 canales a la vez (inmediato). Cambia "Y" por "X" para saltar directo.
angles = [90] * 16
print(cmd("X " + " ".join(f"{a:.2f}" for a in angles)))

In [ ]:
# Escribe tu propio comando
print(cmd("INFO"))

In [ ]:
# ---------- Enviar un CSV por Serial ----------
# Mapeo por posicion de columna: 1a columna de datos -> canal 0, 2a -> canal 1, ... (max 16).
# Una columna de tiempo (t, t_s, time, time_s, t_ms) no se envia: solo marca el ritmo.
import csv, threading

CSV_PATH = "leg_actuator_angles_one_cycle_smooth_n4.csv"   # <- CSV a enviar
RATE_HZ = 50        # solo si el CSV no tiene columna de tiempo
TIME_SCALE = 1.0    # >1 = mas lento, <1 = mas rapido
REPEAT = 1
MAX_CHANNELS = 16
SIGN = 1            # angulo_enviado = SIGN * valor_csv + OFFSET (numero o lista por canal)
OFFSET = 0
TIME_COLUMNS = ("t", "t_s", "time", "time_s", "t_ms")

def per_channel(v, n):
    return list(v) if isinstance(v, (list, tuple)) else [v] * n

def load_csv(path):
    with open(path, newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        rows = [r for r in reader if r and any(c.strip() for c in r)]
    time_idx = next((i for i, h in enumerate(header) if h.strip().lower() in TIME_COLUMNS), None)
    data_idx = [i for i in range(len(header)) if i != time_idx][:MAX_CHANNELS]
    times = None
    if time_idx is not None:
        times = [float(r[time_idx]) for r in rows]
        if header[time_idx].strip().lower() == "t_ms":
            times = [t / 1000 for t in times]
    values = [[float(r[i]) for i in data_idx] for r in rows]
    return [header[i] for i in data_idx], times, values

names, times, values = load_csv(CSV_PATH)
n_ch = len(names)
print(f"{len(values)} filas, {n_ch} canales, tiempo: {'columna' if times else f'{RATE_HZ} Hz fijo'}")
for ch, n in enumerate(names):
    print(f"  canal {ch:2d} <- columna '{n}'")
out = [n for n, col in zip(names, zip(*values)) if min(col) < 0 or max(col) > 180]
if out:
    print(f"\nOJO: {len(out)} columnas salen de 0-180 y el firmware las recortaria: {out}")
    print("Convierte a angulo de servo (SIGN/OFFSET) antes de reproducir en el robot.")

In [ ]:
SIGNS, OFFSETS = per_channel(SIGN, n_ch), per_channel(OFFSET, n_ch)

def row_to_cmd(row, c="X"):
    # Un comando por canal, todos en una sola escritura al puerto
    return "".join(f"{c} {ch} {SIGNS[ch] * v + OFFSETS[ch]:.2f}\n" for ch, v in enumerate(row))

def play(times, values, repeat=1, scale=1.0, rate_hz=50, go_to_start=True):
    stop = threading.Event()
    stats = {"replies": 0, "errors": []}
    def drain():   # lee las respuestas del ESP32 para que no se acumulen
        buf = b""
        while not stop.is_set():
            buf += ser.read(ser.in_waiting or 1)
            while b"\n" in buf:
                line, buf = buf.split(b"\n", 1)
                stats["replies"] += 1
                if line.startswith(b"Error"):
                    stats["errors"].append(line.decode(errors="replace").strip())
    th = threading.Thread(target=drain, daemon=True)
    th.start()
    try:
        if go_to_start:   # rampa suave (Y) a la primera fila
            ser.write(row_to_cmd(values[0], "Y").encode())
            print("Yendo a la posicion inicial...")
            time.sleep(3)
        t0 = time.monotonic()
        period = (times[-1] + (times[1] - times[0])) if times else len(values) / rate_hz
        for rep in range(repeat):
            base = t0 + rep * period * scale
            for k, row in enumerate(values):
                target = base + (times[k] if times else k / rate_hz) * scale
                wait = target - time.monotonic()
                if wait > 0:
                    time.sleep(wait)
                ser.write(row_to_cmd(row).encode())
            print(f"Repeticion {rep + 1}/{repeat} enviada")
    except KeyboardInterrupt:
        print("Detenido por el usuario")
    finally:
        time.sleep(0.3)
        stop.set(); th.join()
    print(f"Respuestas: {stats['replies']}, errores: {len(stats['errors'])}", stats["errors"][:5])

In [ ]:
play(times, values, repeat=REPEAT, scale=TIME_SCALE, rate_hz=RATE_HZ)

In [ ]:
ser.close()